In [50]:
import pandas as pd
import requests
from datetime import datetime
from statsmodels.tsa.seasonal import MSTL

#### I. MTA Daily Ridership and Traffic

In [51]:
import requests

# Use the correct Socrata API endpoint (resource endpoint, not views)
url = "https://data.ny.gov/resource/sayj-mze2.json"

params = {
    "$where": "date >= '2023-01-01'" +
              " AND mode IN ('Bus', 'LIRR', 'MNR', 'Subway')",
    "$order": "date ASC",
    "$limit": 100000
}
response = requests.get(url, params=params)
data = response.json()

In [52]:
# Convert to DataFrame
df = pd.DataFrame(data)
df['date'] = pd.to_datetime(df['date'])
df['count'] = pd.to_numeric(df['count'], errors='coerce')

In [53]:
df.sample(10)

,date,mode,count
314,2023-03-20,MNR,167256
847,2023-07-31,Subway,3204920
1850,2024-04-07,MNR,94103
2433,2024-08-31,LIRR,149890
4092,2025-10-20,Bus,1220586
3556,2025-06-08,Bus,756088
15,2023-01-04,Subway,3413052
19,2023-01-05,Subway,3428520
3189,2025-03-08,LIRR,124078
861,2023-08-04,LIRR,193528


In [54]:
df['mode'].unique()

array(['Bus', 'LIRR', 'MNR', 'Subway'], dtype=object)

In [55]:
abb_to_name = {
    "LIRR": "Long Island Rail Road",
    "Subway": "Subway",
    "Bus": "Bus",
    "MNR": "Metro-North Railroad"
}

In [56]:
df['mode'] = df['mode'].map(abb_to_name)

df['mode'].unique()

array(['Bus', 'Long Island Rail Road', 'Metro-North Railroad', 'Subway'],
      dtype=object)

In [ ]:
def seasonally_adjust_group(group_df):
    """Apply MSTL to a single mode's time series (weekly + yearly seasonality)."""
    group_df = group_df.sort_values('date').set_index('date')
    
    # Ensure continuous daily index
    group_df = group_df.asfreq('D')
    
    # Interpolate any gaps
    group_df['count'] = group_df['count'].interpolate(method='linear')
    
    # MSTL: 7-day (weekly) and 365-day (yearly) seasonality
    mstl = MSTL(group_df['count'], periods=[7, 365])
    result = mstl.fit()
    
    # Remove both seasonal components
    group_df['count_sa'] = group_df['count'] - result.seasonal.sum(axis=1)
    
    return group_df.reset_index()

# Apply seasonal adjustment to each mode
df = (
    df.groupby('mode', group_keys=False)
    .apply(seasonally_adjust_group)
)

TypeError: MSTL.__init__() got an unexpected keyword argument 'robust'

In [38]:
df = df.sort_values(['mode', 'date']).set_index('date')
df['count_ma90'] = df.groupby('mode')['count'].transform(lambda x: x.rolling('90D', min_periods=1).mean())
df = df.reset_index()

In [39]:
# order the df by mode from highest to lowest average daily ridership
mode_order = df.groupby('mode')['count'].mean().sort_values(ascending=False).index
df['mode'] = pd.Categorical(df['mode'], categories=mode_order, ordered=True)
df = df.sort_values(['mode', 'date'])

In [40]:
df.to_csv("daily_ridership.csv", index=False)